# 2023年Java大厂面试｜08：并发编程篇

> 学习目标：把线程基础、并发安全、锁与 AQS、JMM、CAS、线程池、并发容器、ThreadLocal 和项目中的多线程场景串成一套可复述、可落地的面试答案。

**资料范围**：主要依据网盘 `5.2023版Java面试教程/08-并发编程篇/PPT/08-多线程相关面试题.pptx` 在线预览，共 144 页；课件目录另含“代码”“视频”和“在线讲义文稿.txt”（预览大小 61B，未形成可读正文）。本篇按课件问题顺序重排，补充 JDK 版本边界和实践提醒。


## 目录与答题主线

| 模块 | 必会问题 | 记忆主线 |
|---|---|---|
| 线程基础 | 进程/线程、并发/并行、创建、状态、停止 | 执行单元、调度、生命周期 |
| 并发安全 | 原子性、可见性、有序性 | 共享变量为什么会错 |
| 锁与同步器 | `synchronized`、`Lock`、AQS、ReentrantLock、CAS | 互斥、排队、状态、唤醒 |
| 线程池 | 参数、执行流程、队列、拒绝策略、核心线程数 | 控制资源与背压 |
| 并发工具 | CountDownLatch、Semaphore、Future | 等待、限流、异步 |
| 并发容器 | ConcurrentHashMap | JDK 1.7 与 1.8 的实现差异 |
| 线程隔离 | ThreadLocal、弱引用与内存泄漏 | 每线程一份数据，使用后清理 |


## 一、线程基础：先把概念说准

### 1. 进程和线程

- **进程**是正在运行的程序实例，负责装载指令、管理内存和 I/O；不同进程通常拥有相互隔离的地址空间。
- **线程**是进程中的一条指令流。一个进程可以包含多个线程，同一进程内线程共享堆等进程资源，但各自拥有程序计数器、虚拟机栈和部分线程私有数据。
- 线程上下文切换通常比进程切换成本低，但线程越多也会带来调度、栈内存和竞争开销。

面试中不要把“线程共享所有内存”说成绝对事实：共享的是进程级资源，栈帧中的局部变量仍属于各自线程。


### 2. 并发和并行

- **并发（concurrency）**：同一时间段内应对多件事，单核 CPU 通过时间片轮转，微观上仍是串行。
- **并行（parallelism）**：多个 CPU 核心在同一时刻真正执行多个任务。
- 多核机器上并发和并行可以同时存在；并发强调“组织多个任务”，并行强调“同时执行”。

一个好记的比喻：一个人轮流做饭、打扫是并发；两个人分别做两件事是并行。


### 3. 创建线程的四种方式

1. 继承 `Thread`，重写 `run()`；适合演示，受单继承限制。
2. 实现 `Runnable`，交给 `Thread`；任务与线程解耦，通常更推荐。
3. 实现 `Callable<V>`，配合 `FutureTask` 或线程池获得返回值，并能抛出受检异常。
4. 使用线程池提交任务；生产代码应集中管理线程数量、队列和拒绝策略。

```java
class ReportTask implements Runnable {
    @Override public void run() { generateReport(); }
}
new Thread(new ReportTask(), "report-worker").start();

Future<Integer> f = executor.submit(() -> calculate());
Integer result = f.get();
```

`start()` 才会创建新的执行栈并进入调度；直接调用 `run()` 只是普通方法调用，不会开启新线程。


### 4. 线程状态、停止与中断

Java `Thread.State` 包括 `NEW`、`RUNNABLE`、`BLOCKED`、`WAITING`、`TIMED_WAITING`、`TERMINATED`。`RUNNABLE` 包含就绪和运行中两种实际情况。

停止线程应使用**协作式取消**：设置取消标志、调用 `interrupt()`，并在阻塞方法捕获 `InterruptedException` 后恢复中断状态或结束任务。不要使用已废弃的 `stop()`、`suspend()`、`resume()`，它们可能破坏共享状态。

```java
while (!Thread.currentThread().isInterrupted()) {
    doOneUnit();
}

try {
    queue.take();
} catch (InterruptedException e) {
    Thread.currentThread().interrupt(); // 不要吞掉中断信号
    return;
}
```


### 5. 常见协作方法

- `sleep` 让当前线程休眠一段时间，**不会释放已经持有的锁**。
- `wait` 必须在持有同一对象监视器时调用，会释放该对象的监视器，等待 `notify/notifyAll` 或超时/中断后重新竞争锁。
- `notify` 随机唤醒一个等待线程，`notifyAll` 唤醒全部等待线程；唤醒后仍要重新竞争锁，并且通常应使用 `while` 重新检查条件。
- `join` 等待目标线程结束；`yield` 只是提示调度器让出 CPU，不保证效果。

```java
synchronized (lock) {
    while (!ready) lock.wait();
    consume();
}
```

使用 `Condition` 时要配合 `Lock`，它允许为不同条件建立多个等待队列。


## 二、并发安全：三种性质

### 1. 原子性

操作要么全部完成，要么完全不发生，中间状态不能被其他线程观察或打断。`i++` 实际包含读、加一、写回三步，不是原子操作。可以用 `synchronized`、`Lock`、原子类或更合适的数据结构保护它。

### 2. 可见性

一个线程修改共享变量后，其他线程能及时看到最新值。普通变量可能停留在线程工作内存或寄存器中；`volatile`、解锁/加锁、线程启动和结束等建立 happens-before 关系的操作可以保证可见性。

### 3. 有序性

编译器、JIT 和处理器可能在不改变单线程最终语义的前提下重排序。跨线程观察时，如果没有同步约束，可能看到违反代码书写顺序的结果。`volatile` 写读屏障、锁和正确的发布协议可以约束重排序。

> `volatile` 只保证可见性和一定的有序性，不能把复合操作（例如 `count++`）变成原子操作。


## 三、synchronized 与对象监视器

`synchronized` 可以修饰实例方法、静态方法或代码块：实例方法锁 `this`，静态方法锁对应的 `Class` 对象，代码块锁括号中的对象。它具有互斥、可见性和可重入性，退出同步块时自动释放锁。

HotSpot 中同步块与对象监视器相关；竞争激烈时线程可能阻塞和唤醒，代价高于无竞争路径。课件用“偏向锁、轻量级锁、重量级锁”解释锁升级，这是理解历史实现的好入口。**版本勘误**：偏向锁在 JDK 15 起默认禁用并在后续版本移除，面试回答应说明“具体实现随 JDK 版本变化，不要把三种形态当成永久 API 契约”。

```java
private final Object lock = new Object();
public void increment() {
    synchronized (lock) { count++; }
}
```


### synchronized 的底层与使用边界

- 无竞争时 JVM 会走快速路径；竞争时可能进入监视器队列。
- 同一线程可以重复获取同一把 `synchronized` 锁，退出次数要与进入次数匹配。
- 锁对象必须稳定且不可被外部随意替换，避免把字符串常量、可变集合等作为公共锁。
- 同步范围越小越好，但必须覆盖完整的共享状态不变式；只给单条写语句加锁往往解决不了检查与更新之间的竞态。


## 四、Lock 与 ReentrantLock

`Lock` 是接口，常用实现 `ReentrantLock`。相比 `synchronized`，它支持可中断获取、超时获取、公平锁和多个 `Condition`，但必须在 `finally` 中手动释放。

```java
private final ReentrantLock lock = new ReentrantLock();
void update() throws InterruptedException {
    lock.lockInterruptibly();
    try { updateState(); }
    finally { lock.unlock(); }
}
```

`new ReentrantLock()` 默认是非公平锁；公平锁按队列顺序倾向于服务等待者，但吞吐量通常更低。可重入意味着同一线程再次 `lock()` 不会阻塞，必须对应次数调用 `unlock()`。


## 五、AQS：锁和同步器的通用骨架

AQS（`AbstractQueuedSynchronizer`）是构建锁和同步组件的基础框架。核心状态包括：

- `volatile int state`：抽象资源状态，独占锁常用 0/1 表示空闲/占用，可重入锁会累加。
- FIFO 双向等待队列：获取资源失败的线程封装为节点排队，并在前驱释放资源时被唤醒。
- `compareAndSetState`：通过 CAS 原子更新状态，保证并发竞争下只有一个线程成功。

`ReentrantLock`、`Semaphore`、`CountDownLatch` 都基于 AQS，但同步语义不同：锁代表独占或共享资源，信号量代表许可数量，倒计时锁代表等待计数归零。

AQS 本身不是简单等同于“公平锁”。非公平实现允许新线程插队；公平实现会检查队列头节点。具体公平性由同步器实现决定。


## 六、CAS 与原子类

CAS（Compare-And-Set/Swap）以“期望旧值 + 新值”更新共享变量：只有内存值仍等于期望值时才写入新值，否则失败并重试。它是乐观并发控制，常用于 AQS 和 `AtomicInteger` 等 JUC 原子类。

优点是失败线程不会立即阻塞；缺点是高竞争下会自旋消耗 CPU，并存在 ABA 问题（值从 A 变 B 又变回 A）。需要区分版本时可用 `AtomicStampedReference` 等带标记方案。

```java
AtomicInteger n = new AtomicInteger(0);
int next = n.incrementAndGet();
boolean ok = n.compareAndSet(expected, update);
```

CAS 解决的是某个变量的原子更新，不能自动保护多个变量之间的不变式；需要组合操作时仍应使用锁或重新设计数据结构。


## 七、JMM 与 volatile

Java 内存模型（JMM）规定线程对共享内存读写的行为，抽象出主内存和线程工作内存。线程之间不能直接读取彼此的工作副本，必须通过同步关系与主内存交互。

`volatile` 的两个核心语义：

1. 对 volatile 变量的写，对随后读取该变量的线程可见；
2. 在 volatile 读写处建立内存屏障，限制相关指令重排序。

适合用在状态标志、配置快照、一次性安全发布等场景；不适合 `count++`、检查后更新等复合操作。双重检查单例中的实例引用也需要 `volatile`，防止发布过程中发生重排序。

```java
private volatile boolean stopping;
void requestStop() { stopping = true; }
```


## 八、ConcurrentHashMap

`ConcurrentHashMap` 提供线程安全的高效 Map，但复合业务逻辑仍需额外同步或使用 `compute` 等原子方法。

- **JDK 1.7**：`Segment` 数组 + 链表，分段锁底层使用 `ReentrantLock`，不同段可以并发。
- **JDK 1.8**：数组 + 链表/红黑树；空桶插入主要使用 CAS，冲突桶只锁定首节点，粒度更细。树化阈值、扩容和计数等细节随实现版本变化。

```java
map.computeIfAbsent(key, k -> loadValue(k));
```

不要把“线程安全”误解为所有多步操作都自动安全；例如 `if (!map.containsKey(k)) map.put(k,v)` 仍存在竞态，应改用原子复合 API。


## 九、死锁与诊断

死锁通常满足四个条件：互斥、占有且等待、不可剥夺、循环等待。典型错误是两个线程以相反顺序获取 A、B 两把锁。

预防方法：固定锁顺序、缩小锁范围、避免嵌套锁、使用 `tryLock` 超时、减少共享状态。诊断流程：

1. `jps` 找到 Java 进程 ID；
2. `jstack -l <pid>` 查看线程栈和锁等待关系；
3. 结合日志、线程名称和代码定位环路；
4. 生产环境也可使用 JConsole、VisualVM 等可视化工具辅助确认。

`jstack` 是诊断工具，命令参数和输出格式会随 JDK 版本变化，面试时说清“先找进程，再看线程栈和锁依赖”即可。


## 十、线程池核心参数与执行流程

`ThreadPoolExecutor` 的核心构造参数：

| 参数 | 含义 |
|---|---|
| `corePoolSize` | 核心线程数 |
| `maximumPoolSize` | 核心线程 + 临时线程的上限 |
| `keepAliveTime`/`unit` | 临时线程空闲存活时间 |
| `workQueue` | 核心线程繁忙时的任务队列 |
| `threadFactory` | 定制线程名、守护属性、异常处理等 |
| `handler` | 队列和线程都饱和后的拒绝策略 |

提交任务后一般按以下顺序处理：核心线程未满则创建核心线程；核心线程满则入队；队列满且未达到最大线程数则创建临时线程；仍无法接收时执行拒绝策略。线程完成任务后优先从队列取任务，临时线程空闲超时会回收。


### 拒绝策略与阻塞队列

JDK 常见拒绝策略：

- `AbortPolicy`：抛出异常，默认策略；
- `CallerRunsPolicy`：由提交任务的线程执行，形成反压；
- `DiscardOldestPolicy`：丢弃队头旧任务，再尝试提交；
- `DiscardPolicy`：静默丢弃当前任务。

常见队列：

- `ArrayBlockingQueue`：数组、有界、FIFO；
- `LinkedBlockingQueue`：链表，可有界也可无界；
- `SynchronousQueue`：不存储元素，必须直接移交；
- `DelayedWorkQueue`：按延迟时间排序，`ScheduledThreadPoolExecutor` 使用。

队列越大不代表系统越安全：无界队列可能掩盖消费能力不足并最终耗尽内存，应结合任务耗时、流量峰值、超时和监控设置容量。


### 为什么不建议直接使用 Executors

`Executors.newFixedThreadPool` 和 `newSingleThreadExecutor` 使用近似无界的 `LinkedBlockingQueue`，任务堆积可能导致内存压力；`newCachedThreadPool` 的最大线程数接近无限，突发流量可能创建过多线程。生产环境优先显式创建 `ThreadPoolExecutor`，把容量和拒绝策略写在配置或代码中，便于审查和监控。

```java
ExecutorService pool = new ThreadPoolExecutor(
    8, 16, 30, TimeUnit.SECONDS,
    new ArrayBlockingQueue<>(1000),
    namedFactory("order"),
    new ThreadPoolExecutor.CallerRunsPolicy());
```

线程池要配套关闭策略（`shutdown`/`shutdownNow`）、任务超时、指标（活跃数、队列长度、拒绝数）和异常处理。


## 十一、线程数如何估算

课件给出便于面试的经验值：

- I/O 密集型（文件、数据库、网络）：约 `CPU 核数 + 1` 或 `2N + 1`；
- CPU 密集型（计算、图片转换、序列化）：约 `CPU 核数 + 1`；
- 高并发且任务耗时长：先从缓存、拆分服务、扩容和异步化等整体架构解决，再调线程池。

更稳妥的做法是用任务等待时间/计算时间、目标吞吐量和压测结果校准，不把经验公式当作定律。CPU 核数可用 `Runtime.getRuntime().availableProcessors()` 获取。


## 十二、线程池类型与并发工具

### Executors 常见类型（用于理解旧代码）

- `newFixedThreadPool`：固定大小，适合并发上限明确的任务；
- `newSingleThreadExecutor`：单线程、FIFO，适合需要顺序执行的任务；
- `newCachedThreadPool`：按需创建并回收空闲线程，适合短小突发任务但风险高；
- `newScheduledThreadPool`：延迟或周期任务。

### CountDownLatch

初始化计数后，工作线程完成一份任务调用 `countDown()`；协调线程调用 `await()` 等待计数归零。它是一次性的，适合批量导入、等待多个服务准备完成。


### Semaphore 与 Future

`Semaphore` 用许可数限制同时访问某段代码的线程数量，常用于限流、连接池和资源槽位：`acquire()` 获取许可，`release()` 归还许可，必须在 `finally` 中释放。

`Future` 表示异步任务结果，`get()` 会阻塞等待；多个无依赖接口可并发提交后统一收集结果。需要更丰富的组合、超时和异常传播时，可使用 `CompletableFuture`，但仍要指定自定义线程池，避免把业务任务提交到公共线程池。


## 十三、项目多线程场景

### 1. ES 批量导入

千万级数据不要一次性读入内存。先查询总页数，再按页（例如每页 2000 条）创建任务，提交到有界线程池；每个任务导入 ES 后 `countDown()`，主线程 `await()`。注意分页一致性、失败重试、幂等和任务取消，避免用 CountDownLatch 把异常静默吞掉。

### 2. 数据汇总

订单、商品、物流等互不依赖的接口可以并行调用，再汇总 `Future` 结果，把串行耗时 `800ms + 500ms + 500ms` 压缩到接近最长单项耗时。若接口存在依赖，按依赖图分阶段编排，不能盲目并行。

### 3. 异步调用

主流程不需要下游返回值时，可把通知、日志、索引刷新等工作异步提交，缩短响应时间。必须明确可靠性边界：任务是否允许丢失、是否需要持久化消息、失败如何补偿。


## 十四、ThreadLocal

`ThreadLocal` 给每个线程提供独立副本，适合保存事务上下文、租户信息、格式化器或线程绑定的连接资源，避免多个线程争用同一可变对象。

实现上，每个 `Thread` 持有一个 `ThreadLocalMap`；`ThreadLocal` 实例作为 key，值作为 value。`set/get/remove` 都是在当前线程自己的 Map 中定位数据，因此它不是“在线程之间共享数据”的工具。

线程池中的线程会复用，业务任务结束后必须在 `finally` 中 `remove()`，否则旧值可能污染后续请求并造成内存泄漏风险。ThreadLocalMap 的 key 是弱引用、value 是强引用；key 被回收后，value 仍可能滞留，主动清理才是可靠方案。

```java
try {
    USER_CONTEXT.set(userId);
    handle();
} finally {
    USER_CONTEXT.remove();
}
```


## 十五、面试速答模板

- **线程安全的根因？** 共享可变状态被多个线程并发读写，且缺少原子性、可见性或有序性保障。
- **CAS 和锁怎么选？** 单变量短小更新可用 CAS；竞争激烈、临界区较大或需保护多个变量时用锁，并评估阻塞、超时和公平性。
- **synchronized 和 Lock？** 前者语法简单、自动释放、JVM 深度优化；后者 API 更丰富，支持可中断、超时、公平和多个条件变量，但要手动释放。
- **AQS 是什么？** 用 `state` 和 FIFO 等待队列抽象同步资源，ReentrantLock、Semaphore、CountDownLatch 都可基于它实现。
- **线程池怎么设计？** 先识别任务类型和耗时，再定核心/最大线程数、队列容量、拒绝策略、线程命名、超时、监控和关闭方案。
- **ThreadLocal 会泄漏吗？** 线程池线程长期存活，若只 set 不 remove，value 可能长期保留；必须在 finally 清理。


## 十六、自测清单

1. `run()` 和 `start()` 的区别是什么？
2. 如何让 T1、T2、T3 按顺序执行？（提示：`join`、`CountDownLatch`、`Future`）
3. `wait` 为什么必须在同步上下文中调用？它和 `sleep` 谁释放锁？
4. `volatile` 能否解决 `count++`？为什么？
5. AQS 的 `state` 和等待队列各自解决什么问题？
6. `ReentrantLock` 的公平与非公平如何选择？
7. 如何诊断线上死锁？
8. 线程池任务提交时为什么先入队而不是直接创建最大线程？
9. `ArrayBlockingQueue` 和 `LinkedBlockingQueue` 的结构差异是什么？
10. ConcurrentHashMap 的 JDK 1.7、1.8 加锁粒度有什么变化？
11. ThreadLocalMap 为什么使用弱引用 key？为什么仍然要 remove？
12. ES 批量导入和多接口汇总分别适合哪种并发工具？


## 资料说明与勘误

- 课件在线预览标题为 **《08-多线程相关面试题.pptx》**，共 144 页；本篇按页面中的标题、代码和总结页整理。
- `在线讲义文稿.txt` 在网盘列表中显示为 61B，未提供可用正文，因此未把它当作完整讲义引用。
- 课件中的部分代码为教学演示，重复展示或排版断行已在本篇合并；示例未作为可直接运行的完整工程处理。
- 未观看“视频”目录中的课程视频，未运行课件“代码”目录或任何生产工程，未进行压测；线程数、队列容量和超时参数仍需结合实际指标验证。
- 版本提示：偏向锁、JDK 内部锁实现、ConcurrentHashMap 细节会随 JDK 版本变化；回答源码题时先说明目标 JDK，再谈实现。
